# Prepare environment

In [1]:
# Import packages
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import plotly.graph_objects as go


## Prepare dataset

In [2]:
words = "(Lehrkräftebildung OR Lehrerbildung OR Lehrkräfte OR Lehrkräftefortbildung OR Seiteneinstieg OR Quereinstieg OR Lehramt)" # The words you want tweets to include

df = pd.DataFrame()

for filename in os.listdir("data/"):
    if filename.startswith("tweets_lehrkraeftebildung_v2"):
        print(f"Found {filename}")
        df_temp = pd.read_csv(f"data/{filename}")
        #print(df_temp)
        df = pd.concat([df, df_temp])

df.drop_duplicates(inplace=True)

print(f"\nThere are {df.shape[0]} retweets in this dataset.\n")

Found tweets_lehrkraeftebildung_v2_2023-02-16_18-23-50.csv

There are 1670 retweets in this dataset.



In [6]:
print("Here arre a few examples of Tweets:\n")

display(df.text.unique()[:50])

Here arre a few examples of Tweets:



array(['RT @lueckenbildung: Viele Lehrkräfte haben iPads gestellt bekommen. Inwiefern könnten sie mit den Dienstgeräten iMessage zur kollegiumsinte…',
       'RT @SenBJF: Mit dem Ziel, die Digitalkompetenz von Lehrkräften zu stärken, haben wir eine neue #Qualifizierungsreihe gestartet. Seit letzte…',
       'RT @SiemensStiftung: 📢@HPI_DSchool und wir laden zur unsere kostenlosen #Lehrkräfte-Fortbildung „#DesignThinking in #MINT” ein. In einem Mi…',
       'RT @hav_hendrik: Die ersten Bundesländer veröffentlichen Handlungsempfehlungen für Lehrkräfte. Positiv: Es deutet sich an, dass es kein Ver…',
       'RT @AlexSchaumburg: @geierandrea2017 Da steht: Es gibt keine entsprechende VORSCHRIFT. Es hält aber auch niemand Lehrkräfte davon ab, trotz…',
       'RT @AlexHabicher: Parallel zum U15 Dialog an der @UniCologne wollen wir mit Praktiker*innen rund ums Lehramt gute Ideen entwickeln, macht m…',
       'RT @JSchmitzLeipzig: Der #Lehrkräftemangel ist dramatisch. Kinder wie Lehrkräfte stehe

## Calculating the Importance of Users in the Network
The next step is to analyze the network. The extracted tweets and relevant information are returned as a DataFrame containing an 'edge list', or a list of all edges between nodes. The code then calculates three measures of centrality for each user in the network.

- **In-degree centrality** represents the number of edges going into a node. In the case of retweets, centrality will indicate that a user is getting a large number of retweets.
- **Out-degree centrality** represents the number of edges going out of a node. In the case of retweets, centrality will indicate that a user is retweeting a lot.
- **Betweeness centrality**  represents the number of 'shortest paths' between nodes that pass through through a specific node. In the case of tweets, it measures the extent to which a user connects other communities of users.

The code also stores the number of in_degrees (tweets) and out_degrees (retweets) to visualize later with Plotly.

In [35]:
# Create a directed network graph from the DataFrame
G = nx.from_pandas_edgelist(df, "retweeter", "tweeter", create_using=nx.DiGraph())

# Calculate the in-degree centrality for retweets
in_centrality = nx.in_degree_centrality(G)

# Caluculate the total number of tweets for later plotting purposes
in_degrees = dict(G.in_degree())

# Store centralities in DataFrame
popular_tweeters = pd.DataFrame(
    list(in_centrality.items()), columns=["username", "in-degree_centrality"]
)

# Print the most important users by their centrality
popular_tweeters.sort_values("in-degree_centrality", ascending=False).head(20)

,username,in-degree_centrality
338,Deu_Kurier,0.127495
1,lueckenbildung,0.105602
128,leseerlaubnis,0.051513
430,rosenbusch_,0.041854
16,NaN,0.034127
1221,BildungSicher,0.027688
573,SozialarbeitNRW,0.027044
1199,inesschwerdtner,0.019961
324,dm_ms,0.018030
1283,daniela_sepehri,0.017386


In [36]:
# Calculate the out-degree centrality for retweets
out_centrality = nx.out_degree_centrality(G)

# Caluculate the total number of retweets for later plotting purposes
out_degrees = dict(G.out_degree())

# Store centralities in DataFrame
active_retweeters = pd.DataFrame(
    list(out_centrality.items()), columns=["username", "out-degree_centrality"]
)

# Print the most important users by the amount they retweet
active_retweeters.sort_values("out-degree_centrality", ascending=False).head(20)

,username,out-degree_centrality
2,Bot_TwLehrerZ,0.048294
161,Mandala81966,0.003863
329,muchistaken,0.003863
94,twcampus_bot,0.003220
1499,Freihei98604242,0.003220
307,sanjosko,0.003220
636,erik_grundmann,0.003220
462,monchichiundich,0.002576
1197,Ruzzlefuzz,0.002576
285,Freihei89070736,0.002576


In [32]:
# Create an undirected network graph from the DataFrame
G = nx.from_pandas_edgelist(df, "retweeter", "tweeter")

# Calculate the betweenness centrality for retweets
betweetnness = nx.betweenness_centrality(G)

# Store centralities in DataFrame
bridging_users = pd.DataFrame(
    list(betweetnness.items()), columns=["username", "betweenness"]
)

# Print the most important users by how much they bridge other users
bridging_users.sort_values("betweenness", ascending=False).head(20)

,username,betweenness
2,Bot_TwLehrerZ,0.239510
338,Deu_Kurier,0.183694
278,GieslerIris,0.180128
42,KinderJournal,0.179300
103,mrtaanderson,0.176565
1,lueckenbildung,0.164065
377,Venora_Khoon,0.122150
128,leseerlaubnis,0.112768
16,NaN,0.098759
152,News4teachers,0.064152


# Visualizing Follower Networks
The code below creates an interactive network visualization using Plotly. There are a number of different attributes to the plot worth noting:
- The nodes are colored by the number of retweeted tweets. Those with a number of retweeted tweets are colored blue and purple, and those who only retweet are colored yellow.
- Those with more connections in total are larger to make it easier to spot them. 

In [31]:
# Create the graph and specify the layout of the graph
G = nx.from_pandas_edgelist(df, "retweeter", "tweeter")
pos = nx.drawing.layout.spring_layout(G)
nx.set_node_attributes(G, pos, "pos")

# Create a dictionary of nodes and their order
nodes_dict = {id: node for (id, node) in enumerate(G.nodes())}

# Gather edge positions for visualization
edge_x = []
edge_y = []
for edge in G.edges():
    x0_point, y0_point = G.nodes[edge[0]]["pos"]
    x1_point, y1_point = G.nodes[edge[1]]["pos"]
    edge_x.append(x0_point)
    edge_x.append(x1_point)
    edge_x.append(None)
    edge_y.append(y0_point)
    edge_y.append(y1_point)
    edge_y.append(None)

# Add edges as disconnected lines
edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.5, color="#000000"),
    hoverinfo="none",
    mode="lines",
)

# Gather node positions for visualization
node_x = []
node_y = []
for node in G.nodes():
    x, y = G.nodes[node]["pos"]
    node_x.append(x)
    node_y.append(y)

# Iterate through the nodes and store the usernames and tweeting information
node_text = []
node_adjacencies = []
node_sizes = []
node_colors = []

# Iterate through the nodes and create the text to be shown on hover
for node_number, adjacencies in enumerate(G.adjacency()):
    node_text.append(  # Set the text to be shown on hover
        "Username: "
        + str(adjacencies[0])  # The username
        + "<br>"
        + "Number of Connections: "
        + str(len(adjacencies[1]))  # The number of connections
        + "<br>"
        + "Number of Retweeted Tweets: "
        + str(in_degrees[nodes_dict[node_number]])
        + "<br>"
        + "Number of Retweets: "
        + str(out_degrees[nodes_dict[node_number]])
    )
    node_adjacencies.append(len(adjacencies[1]))
    node_sizes.append(len(adjacencies[1]))
    node_colors.append(in_degrees[nodes_dict[node_number]])

# Log transform the color list for visualization purposes
node_colors = np.array(node_colors)
node_colors_temp = np.where(node_colors > 1.0e-10, node_colors, 1.0e-10)
node_colors_log = np.log10(node_colors_temp)

# Scale the size of the nodes between two values
scaler = MinMaxScaler(feature_range=(5, 30))
node_sizes = scaler.fit_transform(np.array(node_sizes).reshape(-1, 1))

# Plot the nodes
node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers",
    hoverinfo="text",
    marker=dict(
        showscale=True,
        colorscale="Plasma",  # For more colorscale options, go here: https://plotly.com/python/builtin-colorscales/
        reversescale=True,
        color=[],
        size=10,
        colorbar=dict(
            thickness=15,
            title="Number of Retweeted Tweets (Log Transformed)",
            titlefont=dict(size=12),
            xanchor="left",
            titleside="right",
            tickvals=[min(node_colors_log), max(node_colors_log)],
            ticktext=["Low", "High"],
        ),
        line=dict(color="Black", width=0.5),
    ),
)

# Set the size, color, and text of the nodes
node_trace.marker.size = node_sizes
node_trace.marker.color = node_colors_log
node_trace.text = node_text

# Customize and display the figure
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="<b>Retweet Network Graph for "
        + str(words)
        + "<b>",  # Set your title here
        title_x=0.5,
        titlefont_size=16,
        showlegend=False,
        paper_bgcolor="#e6e6e6",  # Set the background color (excluding the plot) here
        plot_bgcolor="#e6e6e6",  # Set the plot background color here
        font_color="#000000",
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        width=800,  # Adjust the width of the plot
        height=500,  # Adjust the height of the plot
    ),
)
fig.show()